# Fix the Hinglish summaries: same prompt, larger output limit

Runtime → Change runtime type → **T4 GPU**, then run all cells.

- It re-generates the 75 Hinglish-output summaries (oracle, Trelis and whisper-small × 25 windows), changing only the output limit: 320 → 1024 tokens. Same model, same prompt (read verbatim from your log), greedy decoding.
- **Time:** roughly 45–75 min, because the summaries are now about 3× longer. Each summary is saved to Drive as soon as it's made. If Colab disconnects or the GPU quota runs out, reopen the notebook and run all cells again; it continues where it stopped.
- At the end it prints a check (how many summaries now end cleanly) and downloads `summaries_hinglish_v2.csv`. Send that file back for judging-packet generation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/hinglish_asr_sum'
%cd {ROOT}
!pip install -q bitsandbytes accelerate
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%writefile run_qwen_hinglish_v2.py
"""Re-generate the Hinglish-output summaries with a larger output limit (max_new_tokens 1024).
Identical to Step 2 otherwise: same model + revision, nf4 4-bit, greedy, batch 4, and the
summary_hinglish prompt read VERBATIM from logs/prompts_log.jsonl.
Resumable: each summary is cached in cache/qwen_hinglish_v2/ as soon as it is made.
Writes outputs/step2/summaries_hinglish_v2.csv."""
import datetime as dt, glob, hashlib, json, os, sys
import pandas as pd, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

ROOT = sys.argv[1]
S2 = f"{ROOT}/outputs/step2"; CACHE = f"{ROOT}/cache/qwen_hinglish_v2"; os.makedirs(CACHE, exist_ok=True)
MODEL, REV = "Qwen/Qwen2.5-7B-Instruct", "a09a35458c702b33eeacc393d103063234e8bc28"
MAX_NEW, BATCH = 1024, 4
CONDS = ["oracle", "asr:trelis-hinglish-preview", "asr:whisper-small"]

logged = [json.loads(l) for l in open(f"{ROOT}/logs/prompts_log.jsonl")]
tpl = [d["prompt"] for d in logged if d.get("component") == "summary_hinglish" and d.get("step") == "2-templates"]
assert tpl, "summary_hinglish template not found in prompts_log.jsonl"
SYSTEM, USER_T = tpl[0]["system"], tpl[0]["user_template"]
print("prompt template loaded verbatim from prompts_log.jsonl")

cond = pd.read_csv(f"{S2}/conditions.csv")
wins = sorted(cond.window_id.unique())
jobs = [(w, c, cond[(cond.window_id == w) & (cond.condition == c)].text.iloc[0]) for c in CONDS for w in wins]
key = lambda w, c: hashlib.sha256(f"{w}|{c}|hinglish|v2|{MAX_NEW}".encode()).hexdigest()[:24]
todo = [j for j in jobs if not os.path.exists(f"{CACHE}/{key(j[0], j[1])}.json")]
print(f"{len(jobs) - len(todo)} cached, {len(todo)} to generate")

if todo:
    tok = AutoTokenizer.from_pretrained(MODEL, revision=REV); tok.padding_side = "left"
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(MODEL, revision=REV, quantization_config=bnb,
                                                 device_map="auto", dtype=torch.float16).eval()
    for i in range(0, len(todo), BATCH):
        b = todo[i:i + BATCH]
        prompts = [tok.apply_chat_template([{"role": "system", "content": SYSTEM},
                                            {"role": "user", "content": USER_T.format(transcript=t)}],
                                           tokenize=False, add_generation_prompt=True) for _, _, t in b]
        enc = tok(prompts, return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False, pad_token_id=tok.eos_token_id)
        L = enc["input_ids"].shape[1]
        for k, (w, c, _) in enumerate(b):
            gen = out[k][L:]
            n_gen = int((gen != tok.pad_token_id).sum()) if tok.pad_token_id is not None else len(gen)
            rec = dict(created=dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"), model=MODEL,
                       revision=REV, max_new_tokens=MAX_NEW, batch=BATCH, window_id=w, condition=c,
                       out_lang="hinglish", input_tokens=int(enc["attention_mask"][k].sum()),
                       generated_tokens=n_gen, hit_limit=n_gen >= MAX_NEW,
                       summary=tok.decode(gen, skip_special_tokens=True).strip())
            json.dump(rec, open(f"{CACHE}/{key(w, c)}.json", "w"), ensure_ascii=False)
        print(f"done {min(i + BATCH, len(todo))}/{len(todo)}", flush=True)
    with open(f"{ROOT}/logs/prompts_log.jsonl", "a") as f:
        f.write(json.dumps({"ts": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"), "step": "2-hinglish-v2",
                            "component": "summary_hinglish", "model": MODEL,
                            "params": {"revision": REV, "max_new_tokens": MAX_NEW, "batch": BATCH, "do_sample": False,
                                       "note": "re-run of Hinglish outputs; original max_new_tokens=320 truncated them"},
                            "prompt": {"system": SYSTEM, "user_template": USER_T}}, ensure_ascii=False) + "\n")

df = pd.DataFrame([json.load(open(p)) for p in glob.glob(f"{CACHE}/*.json")])
df["summary_words"] = df.summary.str.split().str.len()
df["ends_clean"] = df.summary.str.strip().str[-1].isin(list('.।!?"\')'))
df.sort_values(["condition", "window_id"]).to_csv(f"{S2}/summaries_hinglish_v2.csv", index=False)
print(df.groupby("condition")[["summary_words", "generated_tokens", "ends_clean", "hit_limit"]].mean().round(2))
old = pd.read_csv(f"{S2}/summaries.csv"); old = old[old.out_lang == "hinglish"].set_index(["window_id", "condition"]).summary
same_start = sum(str(r.summary).startswith(str(old[(r.window_id, r.condition)])[:150]) for r in df.itertuples())
print(f"{same_start}/{len(df)} new summaries start exactly like the old truncated ones (expected under greedy decoding)")


In [ ]:
!python run_qwen_hinglish_v2.py {ROOT}

In [ ]:
from google.colab import files
files.download(f'{ROOT}/outputs/step2/summaries_hinglish_v2.csv')